In [94]:
#@title 儲存格 1：導入
import os
from datetime import datetime
from etils import epath
import functools
from IPython.display import HTML, clear_output
from typing import Any, Dict, Sequence, Tuple, Union
from ml_collections import config_dict

import jax
from jax import numpy as jp
import numpy as np
from flax import struct
from matplotlib import pyplot as plt
import mediapy as media

import mujoco
from mujoco import mjx

from brax import envs
from brax import math
from brax.envs.base import Env, State
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import mjcf

print("所有函式庫已導入。")

所有函式庫已導入。


In [95]:
#@title 儲存格 2：常數與配置

# --- Pupper 常數定義 ---
class _PupperConstants:
    def __init__(self):
        self.PUPPER_ROOT_PATH = epath.Path(os.getcwd())
        self.DEFAULT_XML = self.PUPPER_ROOT_PATH / "scene_mjx2.0.xml"
        self.ROOT_BODY = "torso"
        self.FEET_SITES = ["foot_front_right", "foot_front_left", "foot_hind_right", "foot_hind_left"]
        self.FEET_GEOMS = ["foot_front_right_collision", "foot_front_left_collision", "foot_hind_right_collision", "foot_hind_left_collision"]
        self.JOINT_POS_SENSOR = "joint_pos"
        self.IMU_GYRO_SENSOR = "imu_gyro"
        self.TORSO_QUAT_SENSOR = "torso_quat"
        self.JOINT_VEL_SENSOR = "joint_vel"
        self.ACTUATOR_FORCES_SENSOR = "actuator_forces"
        self.FOOT_CONTACTS_SENSOR = "foot_contacts"

consts = _PupperConstants()
print("Pupper 常數模組已定義。")


# --- Pupper 預設配置 ---
def get_default_config() -> config_dict.ConfigDict:
    config = config_dict.ConfigDict()
    config.sim_dt = 0.002; config.ctrl_dt = 0.02
    config.episode_length = 1000; config.action_repeat = 10
    config.action_scale = 0.3
    config.soft_joint_pos_limit_factor = 0.95
    reward_scales = {
        'tracking_lin_vel': 1.5, 'tracking_ang_vel': 0.8, 'lin_vel_z': -2.0,
        'ang_vel_xy': -0.05, 'orientation': -5.0, 'torques': -0.0002,
        'action_rate': -0.01,
    }
    config.rewards = config_dict.ConfigDict({'scales': config_dict.ConfigDict(reward_scales), 'tracking_sigma': 0.25})
    config.commands = config_dict.ConfigDict({'ranges': {'lin_vel_x': [-0.6, 1.5], 'lin_vel_y': [-0.8, 0.8], 'ang_vel_yaw': [-0.7, 0.7]}, 'zero_command_probability': 0.1})
    return config

print("預設配置函式 get_default_config() 已定義。")

Pupper 常數模組已定義。
預設配置函式 get_default_config() 已定義。


In [96]:
#@title 儲存格 3：完整的 Pupper 環境 (最終版)

class Pupper(Env):
  """
  一個完整的、合併的 Pupper 任務環境，用於非對稱 Actor-Critic 訓練。
  """
  def __init__(self, **kwargs):
    # --- 1. 準備參數 ---
    config = kwargs.pop('config', get_default_config())
    xml_path = kwargs.pop('xml_path', consts.DEFAULT_XML.as_posix())

    # --- 2. 載入模型並創建 Brax System ---
    mj_model = mujoco.MjModel.from_xml_path(xml_path)
    for i, path in enumerate(mj_model.mesh_fpath):
        mj_model.mesh_fpath[i] = os.path.join(consts.PUPPER_ROOT_PATH, path)
    sys = mjcf.load_model(mj_model)

    # --- 3. 使用 sys 物件呼叫父類別建構函式 ---
    super().__init__(sys=sys, **kwargs)
    
    # ========================== 核心修正 ==========================
    # --- 4. 實現抽象屬性 ---
    # backend 屬性直接從 super() 初始化後的 sys 物件中獲取
    self.backend = self.sys.backend

    # action_size 屬性是致動器的數量
    self.action_size = self.sys.nu
    # =============================================================

    # --- 5. 在 super().__init__ 後設定所有自訂屬性 ---
    self._config = config
    self._config.merge(kwargs)
    self.dt = self._config.ctrl_dt
    
    self._mj_model = mj_model
    self._sensor_indices = { s: mjcf.get_sensor(mj_model, s) for s in [
        'joint_pos', 'imu_gyro', 'torso_quat', 'joint_vel', 'actuator_forces', 'foot_contacts'
    ]}
    
    self._init_q = jp.array(self._mj_model.keyframe("home").qpos)
    self._default_pose = self._init_q[7:]
    
    self._torso_body_id = mujoco.mj_name2id(self._mj_model, mujoco.mjtObj.mjOBJ_BODY, consts.ROOT_BODY)
    self._floor_geom_id = mujoco.mj_name2id(self._mj_model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
    self._feet_geom_id = np.array([mujoco.mj_name2id(self._mj_model, mujoco.mjtObj.mjOBJ_GEOM, n) for n in consts.FEET_GEOMS])
    
    self._cmd_ranges = self._config.commands.ranges
    self._zero_cmd_prob = self._config.commands.zero_command_probability

  def reset(self, rng: jax.Array) -> State:
    rng, key_vel, key_cmd = jax.random.split(rng, 3)
    qpos = self._init_q
    qvel = jax.random.uniform(key_vel, (self.sys.nv,), minval=-0.1, maxval=0.1)
    
    data = self.pipeline_init(qpos, qvel)
    
    cmd = self.sample_command(key_cmd)
    info = {"rng": rng, "command": cmd, "last_act": jp.zeros(self.action_size), "feet_air_time": jp.zeros(4)}
    obs_dict = self._get_obs(data, info)
    reward, done = jp.zeros(2)
    metrics = {f"reward/{k}": jp.zeros(()) for k in self._config.rewards.scales.keys()}
    return State(pipeline_state=data, obs=obs_dict, reward=reward, done=done, metrics=metrics, info=info)

  def step(self, state: State, action: jax.Array) -> State:
    data = state.pipeline_state
    motor_targets = self._default_pose + action * self._config.action_scale
    data = self.pipeline_step(data, motor_targets)
    
    def _is_in_contact(geom_id, contacts): return jp.any(contacts.geom1 == geom_id)
    floor_contacts = data.contact[data.contact.geom2 == self._floor_geom_id]
    contact = jax.vmap(_is_in_contact, in_axes=(0, None))(self._feet_geom_id, floor_contacts)

    rng, key_cmd = jax.random.split(state.info['rng'])
    command = self.sample_command(key_cmd)
    info = state.info.copy()
    info.update(rng=rng, command=command, last_act=action,
                feet_air_time=(state.info["feet_air_time"] + self.dt) * (1 - contact))
                
    obs_dict = self._get_obs(data, info)
    rewards_dict = self._get_reward(data, action, info)
    reward = sum(v * self._config.rewards.scales.get(k, 0.0) for k, v in rewards_dict.items())
    done = data.xmat[self._torso_body_id][2, 2] < 0.3
    
    metrics = state.metrics.copy()
    metrics.update({f"reward/{k}": v for k, v in rewards_dict.items()})
    return State(pipeline_state=data, obs=obs_dict, reward=reward, done=done.astype(reward.dtype), metrics=metrics, info=info)

  def _get_obs(self, data: mjx.Data, info: dict) -> Dict[str, jax.Array]:
    joint_angles = self._read_sensor(data, 'joint_pos'); joint_vel = self._read_sensor(data, 'joint_vel')
    projected_gravity = math.rotate(jp.array([0, 0, -1]), self._read_sensor(data, 'torso_quat'))
    
    standard_obs = jp.concatenate([
        self._read_sensor(data, 'imu_gyro'), projected_gravity,
        info['command'], joint_angles - self._default_pose, info['last_act']
    ])
    privileged_obs = jp.concatenate([
        standard_obs, joint_vel, self._read_sensor(data, 'actuator_forces'), self._read_sensor(data, 'foot_contacts')
    ])
    return {'state': standard_obs, 'privileged_state': privileged_obs}

  def _get_reward(self, data: mjx.Data, action: jax.Array, info: dict) -> dict:
      rewards = {}
      torso_vel = data.cvel[self._torso_body_id]; torso_quat = data.xquat[self._torso_body_id]
      local_vel = math.rotate(torso_vel[:3], math.quat_inv(torso_quat))
      projected_gravity = math.rotate(jp.array([0, 0, -1]), self._read_sensor(data, 'torso_quat'))

      rewards['tracking_lin_vel'] = jp.exp(-jp.sum(jp.square(info['command'][:2] - local_vel[:2])) / self._config.rewards.tracking_sigma)
      rewards['tracking_ang_vel'] = jp.exp(-jp.square(info['command'][2] - self._read_sensor(data, 'imu_gyro')[2]) / self._config.rewards.tracking_sigma)
      rewards['orientation'] = -jp.sum(jp.square(projected_gravity[:2]))
      rewards['lin_vel_z'] = -jp.square(local_vel[2])
      rewards['ang_vel_xy'] = -jp.sum(jp.square(self._read_sensor(data, 'imu_gyro')[:2]))
      rewards['torques'] = -jp.sum(jp.square(self._read_sensor(data, 'actuator_forces')))
      rewards['action_rate'] = -jp.sum(jp.square(action - info['last_act']))
      return rewards

  def _read_sensor(self, data: mjx.Data, name: str) -> jax.Array:
    idx = self._sensor_indices[name]
    return data.sensordata[idx.adr:idx.adr + idx.dim]
  
  def sample_command(self, rng: jax.Array) -> jax.Array:
    rng, key_x, key_y, key_z, key_zero = jax.random.split(rng, 5)
    lin_vel_x = jax.random.uniform(key_x, minval=self._cmd_ranges.lin_vel_x[0], maxval=self._cmd_ranges.lin_vel_x[1])
    lin_vel_y = jax.random.uniform(key_y, minval=self._cmd_ranges.lin_vel_y[0], maxval=self._cmd_ranges.lin_vel_y[1])
    ang_vel_yaw = jax.random.uniform(key_z, minval=self._cmd_ranges.ang_vel_yaw[0], maxval=self._cmd_ranges.ang_vel_yaw[1])
    command = jp.array([lin_vel_x, lin_vel_y, ang_vel_yaw])
    return jp.where(jax.random.bernoulli(key_zero, self._zero_cmd_prob), jp.zeros(3), command)

  @property
  def observation_size(self) -> Dict[str, Tuple[int, ...]]:
    return {"state": (33,), "privileged_state": (61,)}

# --- 註冊環境 ---
# 注意：這裡我們不再需要 @property 裝飾器來定義 action_size
if 'pupper' in envs._envs: del envs._envs['pupper']
envs.register_environment('pupper', Pupper)
print("完整的 Pupper 環境已定義並註冊。")

完整的 Pupper 環境已定義並註冊。


In [97]:
#@title 儲存格 4：訓練主程式

# --- 1. 初始化環境 ---
env = envs.get_environment('pupper')
eval_env = envs.get_environment('pupper')
print(f"環境 '{type(env).__name__}' 已成功初始化。")


# --- 2. 定義網路工廠 (Network Factory) ---
make_networks_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(512, 256, 128),
    value_hidden_layer_sizes=(512, 256, 128),
    policy_observation_key='state',
    value_observation_key='privileged_state'
)
print("非對稱 Actor-Critic 網路工廠已定義。")


# --- 3. 定義 PPO 訓練函式 ---
train_fn = functools.partial(
    ppo.train,
    num_timesteps=100_000_000,
    num_envs=2048,
    episode_length=1000,
    normalize_observations=True,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=8,
    batch_size=1024,
    discounting=0.99,
    learning_rate=3e-4,
    entropy_cost=0.01,
    network_factory=make_networks_factory,
    seed=0,
    num_evals=20,
)
print("PPO 訓練函式已配置完成。")


# --- 4. 啟動訓練並視覺化進度 ---
x_data, y_data = [], []
times = [datetime.now()]

def progress(num_steps, metrics):
    times.append(datetime.now())
    x_data.append(num_steps)
    y_data.append(metrics['eval/episode_reward'])
    
    clear_output(wait=True)
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(10, 6))
    plt.plot(x_data, y_data)
    plt.xlabel('Environment Steps')
    plt.ylabel('Average Episode Reward')
    plt.title(f"Training Progress - Current Reward: {y_data[-1]:.2f}")
    plt.show()

print("\n--- 即將開始訓練 ---")

make_inference_fn, params, _ = train_fn(
    environment=env,
    eval_env=eval_env,
    progress_fn=progress
)

print(f"\n--- 訓練完成！---")
print(f"總耗時: {times[-1] - times[0]}")

TypeError: Can't instantiate abstract class Pupper without an implementation for abstract methods 'action_size', 'backend'